# Access & Subset HyP3 SBAS Stack (InSAR or Burst-InSAR)

<br>

This notebook downloads and spatially subsets a Short Baseline Subset (SBAS) stack of interferograms from [ASF HyP3](https://hyp3-docs.asf.alaska.edu/) or [ASF HyP3+](https://hyp3-docs.asf.alaska.edu/hyp3-docs/about/hyp3_plus/), in preparation for processing with MintPy. It assumes that you have already ordered these data via the [ASF Vertex](https://search.asf.alaska.edu/) web app or programmatically with [hyp3-sdk](https://github.com/ASFHyP3/hyp3-sdk).

<hr>

:::{note} **Did you find a bug? Do you have a feature request?**
:::{figure} github_issues.png
:alt: GitHub logo above the word Issues
:width: 10%
:align: left
:target: https://github.com/ASFOpenSARlab/opensarlab_MintPy_Recipe_Book/issues

Explore GitHub Issues on this Jupyter Book's GitHub repository. Find solutions, add to the discussion, or start a new bug report or feature request: <a href="https://github.com/ASFOpenSARlab/opensarlab_MintPy_Recipe_Book/issues" target="_blank" rel="noopener noreferrer">opensarlab_MintPy_Recipe_Book Issues</a>
:::


:::{note} **Have a question related to SAR, ASF data access, or performing SBAS time series analyses with MintPy?**
:::{figure} ASF_support_logo.png
:alt: ASF logo
:width: 10%
:align: left
:target: https://github.com/ASFOpenSARlab/opensarlab_MintPy_Recipe_Book/issues

Contact ASF User Support: <a href="mailto:uso@asf.alaska.edu" target="_blank" rel="noopener noreferrer">uso@asf.alaska.edu</a>
:::

<hr>

## 0. Import Required Software 

In [ ]:
from datetime import datetime
from pathlib import Path
import shutil
import sys
from tqdm.auto import tqdm

import contextily as ctx
import geopandas as gpd
from hyp3_sdk import Batch, HyP3
from ipyfilechooser import FileChooser
from IPython.display import Markdown, display, HTML
import matplotlib.pyplot as plt 
from matplotlib.lines import Line2D
import numpy as np
import opensarlab_lib as osl
from osgeo import gdal, ogr
gdal.UseExceptions()
from shapely.geometry import box
from rasterio.warp import transform_bounds

# Download slurmjob modules
from getpass import getpass
from ipywidgets import IntSlider

current = Path("..").resolve()
sys.path.append(str(current))
import util.util as util

<hr>

## 1. Select or create a working directory for the analysis

In [ ]:
age = osl.select_parameter(
    [
        "Access a new SBAS stack",
        "Add to existing SBAS stack"
    ]
)
display(age)

In [ ]:
# Get data path -- ../data resolves to /work/CBI_InSAR/<project>/data
project_path = Path.cwd().parent
data_path = project_path / "data"

print("Project path: ", project_path)
print("Data folder path: ", data_path)

<hr>

## 2. Migrate SBAS Stack from HyP3

### Select whether you are accessing data from [HyP3 Basic](https://hyp3-docs.asf.alaska.edu/hyp3-docs/about/hyp3_basic/) or [HyP3+](https://hyp3-docs.asf.alaska.edu/hyp3-docs/about/hyp3_plus/)

In [ ]:
hyp3_service_select = osl.select_parameter(
    [
        'HyP3 Basic',
        'HyP3+'
    ]
)
display(hyp3_service_select)

### Create a HyP3 object and authenticate

In [ ]:
hyp3_username = input('NASA Earthdata Login username: ')

In [ ]:
hyp3_password = getpass('NASA Earthdata Login password: ')

In [ ]:
hyp3_select = hyp3_service_select.value

if "Basic" in hyp3_select:
    hyp3 = HyP3(prompt="password")
else:
    hyp3 = HyP3(api_url="https://hyp3-plus.asf.alaska.edu", prompt="password")

### You may search for InSAR projects in your own account or migrate data from any user's account

- Retrieving data from another user's account only requires their username and the project name.
- It does **not** require the other user's password.

In [ ]:
hyp3_project = osl.select_parameter(
    [
        'Access InSAR data with any valid HyP3 username and HyP3 Project Name',
        'Search your Projects for available InSAR data'
    ]
)
display(hyp3_project)

### Select your SBAS stack's HyP3 product type

In [ ]:
product_select = osl.select_parameter(
    [
        'INSAR_GAMMA',
        'INSAR_ISCE_BURST',
        'INSAR_ISCE_MULTI_BURST'
    ]
)
print("Select your SBAS stack's HyP3 product type")
display(product_select)

In [ ]:
product_type = product_select.value

search = "Search" in hyp3_project.value
if search:
    my_hyp3_info = hyp3.my_info()
    active_projects = dict()
    
    print("Checking all HyP3 projects for current INSAR_GAMMA jobs")
    for project in tqdm(my_hyp3_info['job_names']):
            batch = Batch()
            batch = hyp3.find_jobs(
                name=project, 
                job_type=product_type
            ).filter_jobs(running=False, include_expired=False)
            if len(batch) > 0:
                active_projects.update({batch.jobs[0].name: batch})
    
    if len(active_projects) > 0:
        display(Markdown("<text style='color:darkred;'>Note: After selecting a project, you must select the next cell before hitting the 'Run' button or typing Shift/Enter.</text>"))
        display(Markdown("<text style='color:darkred;'>Otherwise, you will rerun this code cell.</text>"))
        print('\nSelect a Project:')
        project_select = osl.select_parameter(active_projects.keys())
        display(project_select)
    else:
        print("Found no active projects containing InSAR products")
else:
    username = input("enter the HyP3 username on the account containing an SBAS stack to migrate")
    project_name = input("Enter the HyP3 project name")
    batch = Batch()
    batch = hyp3.find_jobs(
        name=project_name, 
        job_type=product_type, 
        user_id=username
    ).filter_jobs(running=False, include_expired=False)

### Select a date range of products to migrate

In [ ]:
if search:
    jobs = active_projects[project_select.value]
else:
    jobs = batch

if not "MULTI" in product_type:
    display(Markdown("<text style='color:darkred;'>Note: After selecting a date range, you should select the next cell before hitting the 'Run' button or typing Shift/Enter.</text>"))
    display(Markdown("<text style='color:darkred;'>Otherwise, you may simply rerun this code cell.</text>"))
    print('\nSelect a Date Range:')
    dates = osl.get_job_dates(jobs)
    date_picker = osl.gui_date_picker(dates)
    display(date_picker)

### Save the selected date range and remove products falling outside of it

In [ ]:
if not "MULTI" in product_type:
    date_range = osl.get_slider_vals(date_picker)
    date_range[0] = date_range[0].date()
    date_range[1] = date_range[1].date()
    print(f"Date Range: {str(date_range[0])} to {str(date_range[1])}")
    jobs = osl.filter_jobs_by_date(jobs, date_range)
    
print(f"\nThere are {len(jobs)} products to migrate.")

### Write the download script (slurmjob)

In [ ]:
# Define the number of nodes to be used in the script
num_nodes_slider = IntSlider(
    value=3,
    min=3,
    max=6,
    step=1,
    description="# of nodes:",
    readout_format="d"
)

num_nodes_slider

In [ ]:
# Get slider value
slurmjob_num_nodes = num_nodes_slider.value

slurmjob_num_nodes

In [ ]:
# Get flight paths
osl.set_paths_orbits(jobs)
paths = set()
for p in jobs:
    paths.add(p.path)
display(paths)

In [ ]:
# Write logic for dealing w/ multiple paths -- I'll add this to our Github repo later.
multiple_paths = True
if len(paths) == 1:
    multiple_paths = False
    flight_path = list(paths)[0]
    display(f"Flight path set to {flight_path}. It is the only flight path on this project.")
elif len(paths) > 1:
    path_choice = osl.select_parameter(paths)
    display(path_choice)
else:
    raise Exception("There are 0 flight paths on this project.")

In [ ]:
if multiple_paths:
    flight_path = path_choice.value

In [ ]:
# Get the path to output the script to
slurmjob_download_path = Path.cwd() / "sbas_download.slurm"

slurmjob_download_path

In [ ]:
# Define download script
slurmjob_download_script = \
f"""#!/bin/bash
#SBATCH --job-name=mintpy_job
#SBATCH --output={project_path / "logs"}/mintpy_job_%A_%a.out
#SBATCH --error={project_path / "logs"}/mintpy_job_%A_%a.err
#SBATCH --ntasks={slurmjob_num_nodes} # Number of tasks
#SBATCH --nodes={slurmjob_num_nodes}  # Total number of nodes
#SBATCH --cpus-per-task=4             # Number of CPU cores per task
#SBATCH --time=12:00:00               # Time limit hh:mm:ss

# Define arguments for the Python script
DATA_PATH="{str(data_path)}" # must be under /work/CBI_InSAR
NEW_STACK="{"True" if age.value == "Access a new SBAS stack" else "False"}"
HYP3_USERNAME="{hyp3_username}"
HYP3_PASSWORD="{hyp3_password}"
HYP3_PROJECT_NAME="{project_path.name}"
PRODUCT_TYPE="{product_type}"
DATE_RANGE_START="{str(date_range[0])}" # yyyy-MM-dd
DATE_RANGE_END="{str(date_range[1])}" # yyyy-MM-dd
FLIGHT_PATH={flight_path}

# Run your Python script
srun "$HOME"/.local/envs/opensarlab_mintpy_recipe_book/bin/python $(pwd)/download_sbas_stack.py \
    --data-path $DATA_PATH \
    --new-stack $NEW_STACK \
    --hyp3-username $HYP3_USERNAME \
    --hyp3-password $HYP3_PASSWORD \
    --hyp3-project-name $HYP3_PROJECT_NAME \
    --product-type $PRODUCT_TYPE \
    --date-range $DATE_RANGE_START $DATE_RANGE_END \
    --flight-path $FLIGHT_PATH
"""

print(slurmjob_download_script)

In [ ]:
# Write the contents of the slurmjob_download variable to a file
# WARNING: this will overwrite the existing contents of the slurmjob file for this project, if there is one already!
with open(slurmjob_download_path, "w", encoding="utf-8") as f:
    f.write(slurmjob_download_script)

### Run the download script

On a terminal, log into the HPC node that you want to run the download job from -- this should **not** be the head node (crest-h001)!

Then, navigate to the notebooks folder using the following command...

In [ ]:
# Copy the OUTPUT of this code block and paste it into a terminal to reach the download script's contained folder
print(f"cd {Path.cwd()}")

...and run the following command to run the slurmjob:
```bash
sbatch ./sbas_download.slurm
```

You can monitor the status of the task **after running the command in the last code block** by following along its output using the commands in the following code blocks. To view both the standard and error output, you may need to use a terminal multiplexer like tmux; learn more about it [here](https://www.redhat.com/en/blog/introduction-tmux-linux).

In [ ]:
# Copy the OUTPUT of this code block and paste it into a terminal on the HPC to monitor the migration task's regular output. Type the shortcut Ctrl+C inside the terminal to STOP.
logs_path = project_path / "logs"

print(f"tail -f $(ls -t {logs_path} | grep -E \'mintpy.+.out\' | (echo -n \"{logs_path}/\" && cat) | head -n 1)")

In [ ]:
# Copy the OUTPUT of this code block and paste it into a terminal on the HPC to monitor the migration task's error output. Type the shortcut Ctrl+C inside the terminal to STOP.
logs_path = project_path / "logs"

print(f"tail -f $(ls -t {logs_path} | grep -E 'mintpy.+.err' | (echo -n \"{logs_path}/\" && cat) | head -n 1)")

### Verify downloads
#### Verify that the download script finished executing 
***Do NOT run the following code blocks until the download script (slurmjob) has finished processing. You can make sure that the task finished by:***
1. Checking if there are no slurmjobs running by running this command on the HPC: `squeue -u $USER`
2. Checking if the slurmjob finished gracefully (i.e., in a non-error state) by running this command on the HPC, replacing the ID placeholder with its actual value: `sacct -j <job id> --format=JobID,JobName,Elapsed,State,Start,End`
   1. You can find your job ID by running this command and seeing which job ID matches the start/end dates of your processing time: `sacct --user=<your username> --format=JobID,JobName,Elapsed,State,Start,End`
3. Checking if all the download tasks finished successfully by checking the output log of your task by running the following command, replacing the ID placeholders with their actual values: `cat mintpy_job_<job ID>_<job array ID>.out | grep 'All tasks complete. Exiting.' | wc -l`
   1. The number of tasks complete should match the number of nodes specified in this notebook.
   2. If you don't know the job array ID of your task, it's likely that there is only one file for this job in the project's `logs` folder.

After running these steps, you can be sure that the download script has finished executing.

#### Verify that all the download ZIP files were downloaded correctly
The following code block checks if all the ZIP files for this run were downloaded correctly. If there are no errors here, we can be confident that all of the data products were migrated successfully!

In [ ]:
### Verify that the download script got everything
# Define data folder
data_path = project_path / "data"
for job in jobs:
    # One job may have multiple files, so iterate through that array
    for jobfile in job.files:
        # Get job path
        job_path = data_path / jobfile['filename']
        if job_path.exists():
            file_size = job_path.stat().st_size
            if file_size != jobfile['size']:
                print(f"The size of {jobfile['filename']} on disk ({file_size}) does not match the size from the server ({jobfile['size']}). You may need to redownload it: {jobfile['url']}")
            else:
                print(f"{jobfile['filename']} is OK!")
        else:
            print(f"File {jobfile['filename']} was not downloaded. Restart the slurmjob or download this file manually: {jobfile['url']}")

#### Verify that all the download ZIP files were unzipped correctly
Once we have verified that all the ZIP files have been downloaded, we have to make sure that they were also unzipped correctly. This can be done with the next code block.

*If any files failed to extract, you may extract them manually using the following commmand **from within the project's data folder:*** `unzip <path to zip file>`

In [ ]:
### Verify that the download script extracted all ZIPs
# Define data folder
data_path = project_path / "data"
# Get all ZIP files in data folder
data_path_zips = data_path.glob('./*.zip')
# Check if a corresponding folder exists for that zip file
for zip_file in data_path_zips:
    folder_path = data_path / zip_file.stem
    if folder_path.exists():
        print(f"The image {zip_file.stem} was extracted.")
    else:
        print(f"The image {zip_file.stem} was NOT extracted. You may need to unzip the following file manually:")
        print(f"\t{zip_file}")

<hr>

## 3. Confirm Presence of a DEM, Azimuth Angle Map, and Incidence Angle Map

- These are optional addon products for HyP3, which are necessary for MintPy
    - Incidence angle maps are included with HyP3 jobs when the `Include Look Vectors` option is selected.
    - DEMs are included with HyP3 jobs when the `Include DEM` option is selected
- This is an optional addon product for HyP3, which is necessary for MintPy if running the correct_SET (Solid Earth Tides) step
    - Azimuth angle maps are included with HyP3 jobs when the `Include Look Vectors` option is selected

### All of the above mentioned files will be included in an InSAR project if Set MintPy Options is selected when adding InSAR jobs to a project in ASF-Search (Vertex)

In [ ]:
product_type = product_select.value

dem = sorted(list(data_path.glob('*/*dem*.tif')))
lv_phi = sorted(list(data_path.glob('*/*lv_phi*.tif')))
lv_theta = sorted(list(data_path.glob('*/*lv_theta*.tif')))
water_mask = sorted(list(data_path.glob('*/*_water_mask*.tif')))
unw = sorted(list(data_path.glob('*/*_unw_phase*.tif')))
corr = sorted(list(data_path.glob('*/*_corr*.tif')))
conn_comp = sorted(list(data_path.glob('*/*_conncomp*.tif')))
tiff_path = dem + lv_phi + lv_theta + water_mask + unw + corr + conn_comp

if len(dem) > 0:
    print("Success: Found at least 1 DEM.")
else:
    raise FileNotFoundError("Failed to find at least 1 DEM.\n"
                            "You will not be able to successfully run a MintPy time-series unless you"
                            "reorder your HyP3 project with DEMS or provide one from another source.")
                            
if len(lv_phi) > 0:
    print("Success: Found at least 1 lv_phi look vector file.")
else:
    raise FileNotFoundError("Failed to find at least 1 lv_phi look vector file.\n"
                            "You will not be able to successfully run a MintPy time-series unless your"
                            "reorder your HyP3 project with 'Include Look Vectors' option selected.")
    
if len(lv_theta) > 0:
    print("Success: Found at least 1 lv_theta look vector file.")
else:
    raise FileNotFoundError("Failed to find at least 1 lv_theta look vector file.\n"
                            "You will not be able to successfully run a MintPy time-series unless your"
                            "reorder your HyP3 project with 'Include Look Vectors' option selected.")

<hr>

## 4. Confirm that all data are in the same projection and reproject to the predominant EPSG, if necessary

In [ ]:
gdf = gpd.GeoDataFrame(
    {
    'tiff_path': tiff_path,
    'EPSG': [util.get_epsg(p) for p in tiff_path],
    'geometry': [util.get_geotiff_bbox(p) for p in tiff_path],
    }
)

# check for multiple projections and project to the predominant EPSG 
if gdf['EPSG'].nunique() > 1:
    proj_count = gdf['EPSG'].value_counts()
    predominant_epsg = proj_count.idxmax()
    print(f'reprojecting to predominant EPSG: {predominant_epsg}')
    for _, row in gdf.loc[gdf['EPSG'] != predominant_epsg].iterrows():
        pth = row['tiff_path']
        no_data_val = util.get_no_data_val(pth)
        res = util.get_res(pth)
        
        temp = pth.parent/f"temp_{pth.stem}.tif"
        pth.rename(temp)
        src_epsg = row['EPSG']

        warp_options = {
            "dstSRS":f"EPSG:{predominant_epsg}", "srcSRS":f"EPSG:{src_epsg}",
            "targetAlignedPixels":True,
            "xRes":res, "yRes":res,
            "dstNodata": no_data_val
        }
        gdal.Warp(str(pth), str(temp), **warp_options)
        temp.unlink()

    gdf = gpd.GeoDataFrame(
    {
    'tiff_path': tiff_path,
    'EPSG': [util.get_epsg(p) for p in tiff_path],
    'geometry': [util.get_geotiff_bbox(p) for p in tiff_path],
    }
    )
        
display(gdf)

<hr>

## 4. Subset the Stack

### Select how to define your AOI for subsetting

In [ ]:
subset_option = osl.select_parameter([
    'Subset to area of common coverage, shared by all data in the stack',
    'Provide a polygon in Well-Known Text (WKT)',
    'Provide a shapefile'
])

display(subset_option)

### Determine the maximum and common extents of the stack and input WKT or shapefile path

In [ ]:
from shapely.geometry import Polygon
import shapely.wkt

wkt_poly = 'WKT' in subset_option.value
common_coverage = 'common coverage' in subset_option.value
shapefile = 'shapefile' in subset_option.value

max_extents = osl.get_max_extents(unw)
xmin, ymin, xmax, ymax = transform_bounds(int(osl.get_projection(str(unw[0]))), 3857, *max_extents)
max_extents_3857 = [xmin, ymin, xmax, ymax]

common_extents = osl.get_common_coverage_extents(unw)
xmin, ymin, xmax, ymax = transform_bounds(int(osl.get_projection(str(unw[0]))), 3857, *common_extents)
common_extents_3857 = [xmin, ymin, xmax, ymax]

epsg = int(gdf.iloc[0]['EPSG'])

if shapefile:
    print('Select a shapefile (*.shp)')
    shp_fc = FileChooser(Path.home())
    display(shp_fc)
else:
    correct_wkt_input = False
    while not correct_wkt_input:
        if common_coverage:
            wkt = (f'POLYGON(({common_extents[0]} {common_extents[1]}, {common_extents[2]} {common_extents[1]}, {common_extents[2]} '
                   f'{common_extents[3]}, {common_extents[0]} {common_extents[3]}, {common_extents[0]} {common_extents[1]}))')
            wkt_shapely_geom = shapely.wkt.loads(wkt)
        else:
            wkt, wkt_shapely_geom = util.get_valid_wkt()
            wkt_is_wgs84 = util.possible_wgs84_wkt(wkt)

            if wkt_is_wgs84:
                wkt = util.project_wkt_polygon(wkt, 4326, epsg)
                wkt_shapely_geom = shapely.wkt.loads(wkt)
                            
        wkt_ogr_geom = ogr.CreateGeometryFromWkt(wkt)
        
        if not util.check_within_bounds(wkt_shapely_geom, gdf):
            print('WKT exceeds bounds of at least one dataset')
            if common_coverage:
                raise Exception('Error determining area of common coverage')
            else:
                continue
    
        correct_wkt_input = True
        
    shp_path = data_path / f'shape_{datetime.strftime(datetime.now(), "%Y%m%dT%H%M%S")}.shp'
    util.save_shapefile(wkt_ogr_geom, epsg, shp_path)

### Confirm the AOI location on a plot before subsetting

In [ ]:
%matplotlib inline

display(HTML("""
<div style="
    border-left: 4px solid #4c8bf5;
    background: #f5f8ff;
    padding: 12px;
    margin: 10px 0;
">

<strong>Maximum Extents vs. Common Extents</strong><br><br>

<div style="text-align:center">
  <img src="common_max_stack_coverage.png"
       alt="Image visualizing difference between maximum and common extents"
       width="500">
</div>

<p>
<strong>Maximum Extents:</strong> Pixels in this area are guaranteed to be included
in at least one interferogram in the stack.
</p>

<p>
<strong>Common Extents:</strong> Pixels in this area are guaranteed to included
in every interferogram in the stack.
</p>

</div>
"""))

if shapefile:
    shp_path = Path(shp_fc.selected)
    if shp_path.suffix != '.shp':
        raise Exception(f'Selected file suffix not ".shp"')
  
shp_gdf = gpd.read_file(shp_path)
shp_gdf = shp_gdf.to_crs(crs='EPSG:4326')

box1 = gpd.GeoDataFrame({"geometry": [box(*max_extents)]}, crs=f'EPSG:{epsg}')
box2 = gpd.GeoDataFrame({"geometry": [box(*common_extents)]}, crs=f'EPSG:{epsg}')
box1 = box1.to_crs(crs='EPSG:4326')
box2 = box2.to_crs(crs='EPSG:4326')

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
shp_gdf.plot(ax=ax, color='blue', alpha=0.25, edgecolor='k')
box1.plot(ax=ax, color='none', edgecolor='red', linewidth=4)
box2.plot(ax=ax, color='none', edgecolor='green', linewidth=4)

ctx.add_basemap(ax, crs=shp_gdf.crs.to_string(), source=ctx.providers.Esri.WorldImagery)
ctx.add_basemap(ax, crs=shp_gdf.crs.to_string(), source=ctx.providers.OpenStreetMap.Mapnik, alpha=0.5)

minx, miny, maxx, maxy = box1.total_bounds
buffer_percentage = 0.05
buffer_x = (maxx - minx) * buffer_percentage
buffer_y = (maxy - miny) * buffer_percentage
buffered_minx = minx - buffer_x
buffered_maxx = maxx + buffer_x
buffered_miny = miny - buffer_y
buffered_maxy = maxy + buffer_y

ax.set_xlim(buffered_minx, buffered_maxx)
ax.set_ylim(buffered_miny, buffered_maxy)
ax.set_title('Loaded Shape for Subsetting')

legend_elements = [
    Line2D([0], [0], color='red', lw=4, label='Max area covered by data stack'),
    Line2D([0], [0], color='green', lw=4, label='Common area covered by data stack')
]
ax.legend(handles=legend_elements)
plt.show()

### Subset the data

In [ ]:
for pth in tqdm(gdf['tiff_path']):
    print(f'Subsetting: {pth}')

    if common_coverage:
        temp_pth = pth.parent/f'subset_{pth.name}'
        gdal.Translate(destName=str(temp_pth), srcDS=str(pth), projWin=[common_extents[0], common_extents[3], common_extents[2], common_extents[1]])
        pth.unlink()
        temp_pth.rename(pth)
    else:
        warp_args = gdal.WarpOptions(cutlineDSName=shp_path, cropToCutline=True) 
        gdal.Warp(str(pth), str(pth), options=warp_args)

### Select whether to convert to WGS84 (lat/lon) prior to loading into MintPy

- HyP3 data are delivered in UTM.
- This notebook intentionally only converts to WGS84, not back to UTM.
- Repeatedly converting data back and forth between coordinate systems will lead to a loss in precision.
- If you have already converted the data to WGS84 and wish to go back to UTM, delete the data and rerun this notebook to access fresh copies from HyP3.

In [ ]:
coord_sys_option = osl.select_parameter(['UTM', 'WGS84 (lat/lon)'], 'Select a coordinate system for your data')
display(coord_sys_option)

### If desired, convert to WGS84

In [ ]:
wgs84 = 'WGS84' in coord_sys_option.value
if wgs84:
    for pth in tqdm(gdf['tiff_path']):
        print(f'Converting {pth} to WGS84')
        gdal.Warp(str(pth), str(pth), dstSRS='EPSG:4326')

### Remove any subset scenes containing no data

In [ ]:
removed = []
# check unw_phase for no-data filled rasters and remove parent directories accordingly
for pth in tqdm(gdf.loc[gdf['tiff_path'].astype(str).str.contains('unw_phase', na=False)]['tiff_path']):
    try:
        raster = gdal.Open(str(pth))
        band = raster.ReadAsArray()
    except RuntimeError:
        raise
        
    if np.count_nonzero(band) < 1:
        shutil.rmtree(pth.parent)
        removed.append(pth)

if len(removed) == 0:
    print("No interferograms were removed")
else:
    print(f"{len(removed)} interferograms removed:")
    for pth in removed:
        print(pth.parent)

<hr>

*Author: Alex Lewandowski; Alaska Satellite Facility*